# Today's assignment: Connecting to postgres database
## create a BMC database and create two tables: Teacher and Student

# setting logs

In [ ]:
import logging
import os
import time

import psycopg2
from psycopg2 import OperationalError, Error as Psycopg2Error
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv

logging.basicConfig(
    filename="w2day1.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("internship_logger")
print("Logging configured -> w2day1.log")

In [ ]:
logger.info("Week 2 Day 1 - Started")

# Loading Credentials


In [ ]:
load_dotenv(override=True)  # looks for a ".env" file relative to this notebook

DB_HOST = os.getenv("DB_HOST")       # e.g. "127.0.0.1" (placeholder shown here)
DB_PORT = os.getenv("DB_PORT")       # e.g. "5432"
DB_NAME = os.getenv("DB_NAME")       # e.g. "banking_etl"
DB_USER = os.getenv("DB_USER")       # e.g. "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")  # never printed or committed

_required = {"DB_HOST": DB_HOST, "DB_PORT": DB_PORT, "DB_NAME": DB_NAME,
             "DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}
_missing = [k for k, v in _required.items() if not v]
if _missing:
    raise RuntimeError(
        f"Missing required .env variable(s): {', '.join(_missing)}. "
        f"Please check your .env file."
    )
logger.info("Credentials loaded from .env")
print("Credentials loaded from .env")

In [ ]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.host, self.port, self.dbname, self.user, self.password = (
            host, port, dbname, user, password
        )
        self._connect()

    def _connect(self):
        self.conn = psycopg2.connect(
            host=self.host, port=self.port, dbname=self.dbname,
            user=self.user, password=self.password,
        )
        logger.info("Connected to database '%s'", self.dbname)

    def reconnect(self):
        """Used by the retry decorator when the connection itself has died."""
        try:
            self.conn.close()
        except Exception:
            pass
        self._connect()
        logger.info("Reconnected to database '%s'", self.dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")


def with_retry(max_attempts=3, delay_seconds=1):
    """Retries a db-operation function up to max_attempts times on
    psycopg2.OperationalError, always rolling back (or reconnecting)
    before each retry so the connection never sits in a broken state.
    The decorated function's first positional argument must be the db object.
    """
    def decorator(func):
        def wrapper(db, *args, **kwargs):
            attempt = 0
            while True:
                attempt += 1
                try:
                    return func(db, *args, **kwargs)
                except OperationalError as e:
                    logger.error(
                        "Attempt %d/%d for %s failed: %s",
                        attempt, max_attempts, func.__name__, e,
                    )
                    try:
                        db.rollback()
                    except Exception:
                        # Connection itself is dead (not just the transaction) - reconnect.
                        try:
                            db.reconnect()
                        except Exception as reconnect_err:
                            logger.error("Reconnect failed: %s", reconnect_err)
                    if attempt >= max_attempts:
                        logger.error(
                            "Giving up on %s after %d attempts", func.__name__, max_attempts
                        )
                        raise
                    time.sleep(delay_seconds)
        return wrapper
    return decorator


db = DBConnection(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
print("Connected")


# creating tables Teacher and Student

In [ ]:
DDL_TABLES = """
CREATE TABLE IF NOT EXISTS teacher (
    teacher_id   SERIAL PRIMARY KEY,
    name         VARCHAR(100) NOT NULL,
    subject      VARCHAR(50),
    email        VARCHAR(100) UNIQUE
);

CREATE TABLE IF NOT EXISTS student (
    student_id   SERIAL PRIMARY KEY,
    name         VARCHAR(100) NOT NULL,
    grade        VARCHAR(10),
    email        VARCHAR(100) UNIQUE,
    teacher_id   INTEGER REFERENCES teacher(teacher_id)
);
"""


@with_retry()
def create_schema(db):
    try:
        with db.cursor() as cur:
            cur.execute(DDL_TABLES)
        db.commit()
        logger.info("Schema created/verified (teacher and student tables)")
        print("Schema created/verified")
    except Psycopg2Error as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise


create_schema(db)

# seed data


In [ ]:
def truncate_all(db):
    """Wipes existing rows so this notebook is safely re-runnable."""
    with db.cursor() as cur:
        cur.execute(
            "TRUNCATE TABLE student, teacher RESTART IDENTITY CASCADE;"
        )
    db.commit()
    logger.info("Truncated teacher and student before reseeding")


def seed_teachers_and_students(db):
    with db.cursor() as cur:
        cur.executemany(
            "INSERT INTO teacher (name, subject, email) VALUES (%s, %s, %s);",
            [
                ("Ram Sharma", "Mathematics", "ram.sharma@example.com"),
                ("Sita Koirala", "Science", "sita.koirala@example.com"),
                ("Hari Thapa", "English", "hari.thapa@example.com"),
            ],
        )
        cur.executemany(
            "INSERT INTO student (name, grade, email, teacher_id) VALUES (%s, %s, %s, %s);",
            [
                ("Anish Gurung", "10", "anish.gurung@example.com", 1),
                ("Priya Maharjan", "10", "priya.maharjan@example.com", 1),
                ("Kiran Rai", "9", "kiran.rai@example.com", 2),
                ("Sarita Lama", "9", "sarita.lama@example.com", 2),
                ("Bibek Shrestha", "11", "bibek.shrestha@example.com", 3),
            ],
        )
    db.commit()
    logger.info("Seeded 3 teachers and 5 students")
    print("Seed data inserted: 3 teachers, 5 students")


try:
    truncate_all(db)
    seed_teachers_and_students(db)
    print("All data loaded successfully.")
except Psycopg2Error as e:
    db.rollback()
    logger.error("Data load failed: %s", e)
    raise

## Verify

Joins `student` to `teacher` and prints each student with their
assigned teacher, plus a count check.

In [ ]:
@with_retry()
def verify(db):
    with db.cursor() as cur:
        cur.execute("SELECT COUNT(*) AS n FROM teacher;")
        n_teachers = cur.fetchone()["n"]

        cur.execute("SELECT COUNT(*) AS n FROM student;")
        n_students = cur.fetchone()["n"]

        cur.execute("""
            SELECT s.student_id, s.name AS student_name, s.grade,
                   t.name AS teacher_name, t.subject
            FROM student s
            JOIN teacher t ON s.teacher_id = t.teacher_id
            ORDER BY s.student_id;
        """)
        rows = cur.fetchall()

    print(f"teachers: {n_teachers}  |  students: {n_students}\n")
    print(f"{'ID':<4}{'Student':<18}{'Grade':<8}{'Teacher':<16}{'Subject':<12}")
    for r in rows:
        print(f"{r['student_id']:<4}{r['student_name']:<18}{r['grade']:<8}"
              f"{r['teacher_name']:<16}{r['subject']:<12}")

    logger.info("Verification: %d teachers, %d students", n_teachers, n_students)
    return rows


_ = verify(db)

In [ ]:
logger.info("W2D1 completed successfully")